In [5]:
# =============================================================================
# [FILE 3-v4] miryang_sensitivity_analysis_v4.py
# 가중치 민감도 분석 — 심사자 1·3번 대응
#
# 설계:
#   STEP A. 원본 사고 데이터 → 읍면동×연도×월별 12유형 건수 집계
#           → 가중치 시나리오(±10/20/30%)별 traffic_weight 재산출
#   STEP B. 독립변수 파일과 병합 → LightGBM 재학습 → AUC/순위 비교
#
# 입력:
#   - 밀양_사고_원본.csv
#   - 독립변수_추가_0911.csv
#
# 출력:
#   TableS1_Sensitivity_AUC.csv
#   TableS2_Sensitivity_RankOverlap.csv
#   TableS3_Sensitivity_Correlation.csv
#   FigS1_Sensitivity_AUC.png
#   FigS2_Sensitivity_Heatmap.png
# =============================================================================

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from lightgbm import LGBMClassifier

plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 11,
                     'figure.dpi': 150})
SEED = 42

# =============================================================================
# 가중치 테이블 (논문 Fig.3)
# final_w = expert_mean × (1+δ) + preset_w
# =============================================================================
WEIGHT_DF = pd.DataFrame([
    ('경상',    '차대차',    7.0,  3),
    ('경상',    '차대사람',  8.0,  4),
    ('경상',    '차량단독',  6.3,  6),
    ('부상신고', '차대차',   5.3,  3),
    ('부상신고', '차대사람', 6.3,  5),
    ('부상신고', '차량단독', 8.0, 10),
    ('중상',    '차대차',    8.7, 20),
    ('중상',    '차대사람',  7.3,  4),
    ('중상',    '차량단독',  9.3, 30),
    ('사망',    '차대차',    8.0, 20),
    ('사망',    '차대사람', 10.0, 50),
    ('사망',    '차량단독',  8.0, 20),
], columns=['severity', 'vtype', 'expert_mean', 'preset_w'])

# =============================================================================
# STEP A-1. 원본 사고 데이터 로드
# =============================================================================
print("=" * 60)
print("STEP A-1 | 원본 사고 데이터 로드")
print("=" * 60)

RAW_PATH = './밀양_사고_원본.csv'
IND_PATH = './독립변수_추가_0911.csv'

df_acc = pd.read_csv(RAW_PATH, encoding='utf-8-sig')
print(f"  원본 사고 건수: {len(df_acc)}")

# 연도·월 추출
df_acc['연도'] = df_acc['사고일시'].str.extract(r'(\d{4})').astype(int)
df_acc['월']   = df_acc['사고일시'].str.extract(r'\d{4}년\s*(\d+)월').astype(int)

# 읍면동: 시군구 마지막 토큰
df_acc['읍면동'] = df_acc['시군구'].str.split().str[-1].str.strip()

# 대분류 매핑
def map_severity(x):
    x = str(x)
    if '사망'   in x: return '사망'
    if '중상'   in x: return '중상'
    if '부상신고' in x: return '부상신고'
    return '경상'

# 종분류 매핑
def map_vtype(x):
    x = str(x)
    if '차대사람' in x: return '차대사람'
    if '차량단독' in x: return '차량단독'
    return '차대차'

df_acc['severity'] = df_acc['사고내용'].apply(map_severity)
df_acc['vtype']    = df_acc['사고유형'].apply(map_vtype)

# =============================================================================
# STEP A-2. 읍면동×연도×월×12유형 건수 집계
# =============================================================================
print("\nSTEP A-2 | 12유형 건수 집계 (월 단위)")

df_counts = (df_acc
    .groupby(['읍면동','연도','월','severity','vtype'])
    .size()
    .reset_index(name='n'))

# wide pivot
df_wide = df_counts.pivot_table(
    index=['읍면동','연도','월'],
    columns=['severity','vtype'],
    values='n',
    aggfunc='sum',
    fill_value=0
).reset_index()

# MultiIndex 컬럼 평탄화
if isinstance(df_wide.columns, pd.MultiIndex):
    df_wide.columns = [
        f"{a}_{b}".strip('_') if b else a
        for a, b in df_wide.columns.to_flat_index()
    ]
else:
    df_wide.columns = [str(c) for c in df_wide.columns]

print(f"  컬럼 확인: {df_wide.columns.tolist()[:8]}")

# 12유형 컬럼 매핑 — 없으면 0으로 채움
TYPE_COLS = {}
for _, row in WEIGHT_DF.iterrows():
    col = f"{row['severity']}_{row['vtype']}"
    if col not in df_wide.columns:
        df_wide[col] = 0
    TYPE_COLS[(row['severity'], row['vtype'])] = col

# 키 str 통일
for c in ['읍면동', '연도', '월']:
    df_wide[c] = df_wide[c].astype(str).str.strip()

print(f"  집계 완료: {df_wide.shape}")
print(f"  읍면동 샘플: {sorted(df_wide['읍면동'].unique())[:5]}")
print(f"  연도 값:    {sorted(df_wide['연도'].unique())}")
print(f"  월 샘플:    {sorted(df_wide['월'].unique())[:6]}")

# =============================================================================
# STEP A-3. 독립변수 파일 로드 & 병합
# =============================================================================
print("\nSTEP A-3 | 독립변수 파일 로드")

df_ind = pd.read_csv(IND_PATH, encoding='utf-8-sig')
print(f"  독립변수 shape: {df_ind.shape}")

# 병합 키 추출 & str 통일
df_ind['_읍면동'] = df_ind['시군구_읍면동명'].astype(str).str.strip()
df_ind['_연도']   = df_ind['사고일시_연도'].astype(str).str.extract(r'(\d{4})')[0].str.strip()
df_ind['_월']     = df_ind['사고일시_월'].astype(str).str.extract(r'(\d+)')[0].str.strip()

print(f"  읍면동 샘플(ind): {sorted(df_ind['_읍면동'].unique())[:5]}")
print(f"  연도 샘플(ind):   {sorted(df_ind['_연도'].unique())}")
print(f"  월 샘플(ind):     {sorted(df_ind['_월'].unique())[:6]}")

# 키 교집합 확인
emd_ind  = set(df_ind['_읍면동'].unique())
emd_wide = set(df_wide['읍면동'].unique())
print(f"  읍면동 매칭: {len(emd_ind & emd_wide)}/{len(emd_ind)}  "
      f"미매칭: {emd_ind - emd_wide}")

# 12유형 건수 병합 (월 단위)
merge_cols = ['읍면동', '연도', '월'] + list(TYPE_COLS.values())
df_type_counts = df_ind[['_읍면동', '_연도', '_월']].copy()
df_type_counts = df_type_counts.merge(
    df_wide[merge_cols],
    left_on=['_읍면동', '_연도', '_월'],
    right_on=['읍면동', '연도', '월'],
    how='left'
).fillna(0)

matched_rate = (df_type_counts[list(TYPE_COLS.values())].sum(axis=1) > 0).mean()
print(f"  행 단위 매칭률: {matched_rate:.1%}")

# =============================================================================
# STEP A-4. 시나리오별 TRI 재산출
# =============================================================================
print("\nSTEP A-4 | 가중치 시나리오별 TRI 재산출")

DELTA_LIST = [-0.30, -0.20, -0.10, 0.00, +0.10, +0.20, +0.30]
LABEL_MAP  = {d: ('Base (0%)' if d == 0 else f'{d:+.0%}') for d in DELTA_LIST}

tri_scenarios = {}
for delta in DELTA_LIST:
    label = LABEL_MAP[delta]
    tri   = pd.Series(0.0, index=df_type_counts.index)
    for _, wrow in WEIGHT_DF.iterrows():
        col   = TYPE_COLS[(wrow['severity'], wrow['vtype'])]
        w_new = wrow['expert_mean'] * (1 + delta) + wrow['preset_w']
        tri  += df_type_counts[col].astype(float) * w_new
    tri_scenarios[label] = tri
    print(f"  {label:<14}  mean={tri.mean():.2f}  "
          f"std={tri.std():.2f}  median={tri.median():.2f}")

# 재산출 Base vs 논문 원본 비교
base_tri  = tri_scenarios['Base (0%)']
orig_tri  = df_ind['traffic_weight'].astype(float)
sr, _     = spearmanr(base_tri, orig_tri)
print(f"\n  재산출 Base vs 논문 원본 Spearman r = {sr:.4f}")

# =============================================================================
# STEP B. 독립변수 X 준비
# =============================================================================
print("\n" + "=" * 60)
print("STEP B | 모델 학습용 X 준비")
print("=" * 60)

drop_cols = [
    '시군구_시군명', '사고일시_연도', '사고일시_월', '사고일시_분기',
    '시군구_읍면동명', 'traffic_weight', '법규위반_불법유턴_건수',
    '_읍면동', '_연도', '_월', '읍면동', '연도', '월'
]
X = df_ind.drop(columns=[c for c in drop_cols if c in df_ind.columns])
X = X.select_dtypes(include=[np.number]).fillna(X.mean())
print(f"  X shape: {X.shape}")

# =============================================================================
# STEP C. 시나리오별 LightGBM 실험
# =============================================================================
print("\n" + "=" * 60)
print("STEP C | 시나리오별 실험")
print("=" * 60)

results      = []
rank_records = {}
emd_series   = df_ind['시군구_읍면동명'].astype(str).str.strip()

for delta in DELTA_LIST:
    label  = LABEL_MAP[delta]
    y_new  = tri_scenarios[label]
    thresh = y_new.median()
    y_cls  = (y_new >= thresh).astype(int)

    if y_cls.nunique() < 2:
        print(f"  [SKIP] {label} — 단일 클래스")
        continue

    X_tr, X_te, yc_tr, yc_te = train_test_split(
        X, y_cls, test_size=0.2, random_state=SEED, stratify=y_cls)

    clf = LGBMClassifier(n_estimators=100, max_depth=2,
                          random_state=SEED, n_jobs=-1, verbose=-1)
    clf.fit(X_tr, yc_tr)
    yp    = clf.predict(X_te)
    yprob = clf.predict_proba(X_te)[:, 1]

    # 5-Fold CV AUC
    skf    = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    cv_auc = []
    for tr_i, va_i in skf.split(X, y_cls):
        if y_cls.iloc[va_i].nunique() < 2:
            continue
        m2 = LGBMClassifier(n_estimators=100, max_depth=2,
                              random_state=SEED, n_jobs=-1, verbose=-1)
        m2.fit(X.iloc[tr_i], y_cls.iloc[tr_i])
        cv_auc.append(roc_auc_score(y_cls.iloc[va_i],
                                     m2.predict_proba(X.iloc[va_i])[:, 1]))

    auc = round(roc_auc_score(yc_te, yprob), 4)
    f1  = round(f1_score(yc_te, yp), 4)
    acc = round(accuracy_score(yc_te, yp), 4)

    results.append({
        'Weight Scenario': label,
        'Threshold (TRI)': round(thresh, 2),
        'High-risk N'    : int(y_cls.sum()),
        'ROC-AUC'        : auc,
        'F1-Score'       : f1,
        'Accuracy'       : acc,
        'CV AUC (mean)'  : round(np.mean(cv_auc), 4),
        'CV AUC (std)'   : round(np.std(cv_auc), 4),
    })

    # 고위험 읍면동 Top-10
    df_rank = pd.DataFrame({'읍면동': emd_series.values,
                             'prob': clf.predict_proba(X)[:, 1]})
    top10 = (df_rank.groupby('읍면동')['prob']
                     .mean()
                     .sort_values(ascending=False)
                     .head(10).index.tolist())
    rank_records[label] = top10

    print(f"  {label:<14}  thresh={thresh:.1f}  "
          f"AUC={auc:.4f}  F1={f1:.4f}  "
          f"CV={np.mean(cv_auc):.4f}±{np.std(cv_auc):.4f}")

# =============================================================================
# STEP D. 순위 안정성 & 상관 분석
# =============================================================================
print("\n" + "=" * 60)
print("STEP D | 순위 안정성 & 상관 분석")
print("=" * 60)

base_top10   = rank_records.get('Base (0%)', [])
overlap_rows = []
for label, top10 in rank_records.items():
    ov = len(set(top10) & set(base_top10))
    overlap_rows.append({
        'Scenario'   : label,
        'Overlap'    : ov,
        'Overlap (%)': ov * 10,
        'Top-10'     : ', '.join(top10),
    })
    print(f"  {label:<14}  Overlap={ov}/10  ({ov*10}%)")

corr_rows = []
base_arr  = tri_scenarios['Base (0%)'].values
for label, tri in tri_scenarios.items():
    sr, _ = spearmanr(base_arr, tri.values)
    corr_rows.append({
        'Scenario'  : label,
        'Spearman r': round(sr, 4),
        'Mean TRI'  : round(tri.mean(), 2),
        'Std TRI'   : round(tri.std(), 2),
    })

# =============================================================================
# STEP E. 저장
# =============================================================================
print("\n" + "=" * 60)
print("STEP E | 저장")
print("=" * 60)

pd.DataFrame(results).to_csv(
    './TableS1_Sensitivity_AUC.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(overlap_rows).to_csv(
    './TableS2_Sensitivity_RankOverlap.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(corr_rows).to_csv(
    './TableS3_Sensitivity_Correlation.csv', index=False, encoding='utf-8-sig')
print("  [Saved] TableS1~S3")

# =============================================================================
# STEP F. 시각화
# =============================================================================
print("\nSTEP F | 시각화")

if results:
    labels = [r['Weight Scenario'] for r in results]
    aucs   = [r['ROC-AUC']         for r in results]
    f1s    = [r['F1-Score']        for r in results]
    cv_m   = [r['CV AUC (mean)']   for r in results]
    cv_s   = [r['CV AUC (std)']    for r in results]
    x      = np.arange(len(labels))

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(x, aucs, marker='o', lw=2.5, color='steelblue', label='ROC-AUC (Test)')
    ax.fill_between(x,
                    [m - s for m, s in zip(cv_m, cv_s)],
                    [m + s for m, s in zip(cv_m, cv_s)],
                    alpha=0.15, color='steelblue', label='CV AUC ± std')
    ax.plot(x, cv_m, marker='s', lw=1.5, ls='--',
            color='steelblue', alpha=0.6, label='CV AUC (mean)')
    ax.plot(x, f1s, marker='^', lw=2.0, color='tomato', label='F1-Score')
    if 'Base (0%)' in labels:
        ax.axvline(labels.index('Base (0%)'), color='gray', ls=':', lw=1.5)
    for i, (a, f) in enumerate(zip(aucs, f1s)):
        ax.text(i, a + 0.004, f'{a:.4f}', ha='center', fontsize=8, color='steelblue')
        ax.text(i, f - 0.014, f'{f:.4f}', ha='center', fontsize=8, color='tomato')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=20, ha='right')
    ax.set_ylabel('Score'); ax.set_ylim(0.75, 1.05)
    ax.grid(axis='y', ls='--', alpha=0.4)
    ax.legend(fontsize=9)
    ax.set_title(
        f'AUC range: {max(aucs)-min(aucs):.4f}  |  '
        f'F1 range: {max(f1s)-min(f1s):.4f}', fontsize=10)
    plt.tight_layout()
    plt.savefig('./FigS1_Sensitivity_AUC.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("  [Saved] FigS1_Sensitivity_AUC.png")

if rank_records:
    all_emds = sorted(set(e for t in rank_records.values() for e in t))
    s_labels = list(rank_records.keys())
    mat = np.zeros((len(all_emds), len(s_labels)), dtype=int)
    for j, lb in enumerate(s_labels):
        for i, em in enumerate(all_emds):
            if em in rank_records[lb]:
                mat[i, j] = rank_records[lb].index(em) + 1
    fig, ax = plt.subplots(figsize=(max(9, len(s_labels) * 1.5),
                                     max(5, len(all_emds) * 0.55)))
    mask0 = (mat == 0)
    sns.heatmap(mat, annot=True, fmt='d', cmap='YlOrRd_r', mask=mask0,
                linewidths=0.5, linecolor='white',
                xticklabels=s_labels, yticklabels=all_emds, ax=ax,
                cbar_kws={'label': 'Risk Rank (1=Highest)'})
    sns.heatmap(mat, annot=False, cmap=['#eeeeee'], mask=~mask0,
                linewidths=0.5, linecolor='white',
                xticklabels=s_labels, yticklabels=all_emds,
                ax=ax, cbar=False)
    ax.set_xlabel('Weight Scenario')
    ax.set_ylabel('Sub-district (읍면동)')
    ax.tick_params(axis='x', rotation=25)
    plt.tight_layout()
    plt.savefig('./FigS2_Sensitivity_Heatmap.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("  [Saved] FigS2_Sensitivity_Heatmap.png")

# =============================================================================
# STEP G. 최종 요약
# =============================================================================
print("\n" + "=" * 60)
print("SUMMARY (논문 본문 삽입용)")
print("=" * 60)

if results:
    aucs = [r['ROC-AUC'] for r in results]
    f1s  = [r['F1-Score'] for r in results]
    ovs  = [r['Overlap']  for r in overlap_rows]
    min_sr = min(r['Spearman r'] for r in corr_rows
                 if r['Scenario'] != 'Base (0%)')
    print(f"""
  ROC-AUC : {min(aucs):.4f} ~ {max(aucs):.4f}  (변동 폭 {max(aucs)-min(aucs):.4f})
  F1-Score: {min(f1s):.4f} ~ {max(f1s):.4f}  (변동 폭 {max(f1s)-min(f1s):.4f})
  Top-10 평균 일치율: {np.mean(ovs)*10:.0f}%
  Spearman r 최솟값: {min_sr:.4f}

  ▶ 논문 본문 기술 예시:
  전문가 평가 점수(expert_mean)를 ±30% 범위에서 변동시킨
  7개 시나리오에서 LightGBM을 재학습한 결과, ROC-AUC는
  {min(aucs):.4f}~{max(aucs):.4f} (변동 폭 {max(aucs)-min(aucs):.4f}),
  고위험 읍면동 Top-10 평균 일치율은 {np.mean(ovs)*10:.0f}%,
  위험지수 Spearman r 최솟값은 {min_sr:.4f}로 나타나
  가중치 불확실성에도 고위험 지역 탐지 결과가 안정적임을 확인하였다.
    """)

STEP A-1 | 원본 사고 데이터 로드
  원본 사고 건수: 1459

STEP A-2 | 12유형 건수 집계 (월 단위)
  컬럼 확인: ['읍면동', '연도', '월', '경상_차대사람', '경상_차대차', '경상_차량단독', '부상신고_차대사람', '부상신고_차대차']
  집계 완료: (447, 15)
  읍면동 샘플: ['가곡동', '교동', '남포동', '내이동', '내일동']
  연도 값:    ['2020', '2021', '2022']
  월 샘플:    ['1', '10', '11', '12', '2', '3']

STEP A-3 | 독립변수 파일 로드
  독립변수 shape: (447, 40)
  읍면동 샘플(ind): ['가곡동', '교동', '남포동', '내이동', '내일동']
  연도 샘플(ind):   ['2020', '2021', '2022']
  월 샘플(ind):     ['1', '10', '11', '12', '2', '3']
  읍면동 매칭: 19/19  미매칭: set()
  행 단위 매칭률: 100.0%

STEP A-4 | 가중치 시나리오별 TRI 재산출
  -30%            mean=49.01  std=41.39  median=35.69
  -20%            mean=51.52  std=43.32  median=37.36
  -10%            mean=54.02  std=45.26  median=38.47
  Base (0%)       mean=56.52  std=47.21  median=40.70
  +10%            mean=59.02  std=49.17  median=42.37
  +20%            mean=61.52  std=51.14  median=44.00
  +30%            mean=64.02  std=53.12  median=45.50

  재산출 Base vs 논문 원본 Spearman r = 0.9198

STEP B | 모델 학